# XAI - Is a signature's ranking specific? (Analysis 1) - TP vs TN, all targets

**Question.** Are the patches most important for a signature `T` *its own*, or the same ones every
other signature uses? And are they more specific when the model **detects it correctly**?

**Design.** No null model: the between-group comparison is itself the control. **Primary contrast
TP vs TN** (model correctly present vs correctly absent), **secondary plus vs minus** (biological
presence). Status = group (true-top3) x eligible (pred-top3).

**Primary metric - Spearman.** Per patient, mean correlation between T's score ranking and that of
every other signature. Specific means low correlation, expected **lower in TP** (MWU TP<TN).

**Secondary (top-K).** Jaccard (expected TP<TN) and uniqueness (expected TP>TN).

Runs on 4 pooled folds (pooled cache), all COAD targets (SBS18/44/93) and STAD (SBS8/18/40a/44).


## 0. Config

In [ ]:
import os, math, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import mannwhitneyu
import pyarrow.parquet as pq
from tqdm.auto import tqdm

GRAD_DIR  = '../grad'
OUT_DIR   = os.path.join(GRAD_DIR, 'xai_outputs')
SPLIT_DIR = '../coad_stad_explainability'
FOLD_DIR  = '../grad_cam_signatures_tcga'   # parquet grad-score per fold (per costruire le cache mancanti)
os.makedirs(OUT_DIR, exist_ok=True)

SCORE_COL = 'grad_score'
K_LIST    = [10, 20, 50, 100]
SEED      = 42
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

# cache read: only the useful columns + category dtype (repeated strings -> codes:
# memoria da decine di GB a ~1 GB; risolve il kernel death sullo Spearman).
USECOLS  = ['sample_id', 'case_id', 'signature_name', 'patch_index', 'grad_score', 'attention_weight']
CAT_COLS = ['sample_id', 'case_id', 'signature_name']
SCORES   = ['grad_score', 'combined_score']  # ranking: grad puro vs combined = attn*ReLU(grad)
PLOT_PER_TARGET = False   # False = niente figure per singolo target (molto piu' veloce)

colors = {'TP': '#2ca02c', 'FN': '#ff7f0e', 'TN': '#d62728', 'FP': '#9467bd'}
# Contrasti definiti sullo STATO (TP/FN/TN/FP, da group x eligible):
#   primario  = TP vs TN  (modello CORRETTAMENTE presente vs CORRETTAMENTE assente)
#   secondario= plus vs minus (presenza biologica, ignora la correttezza del modello)
GROUP_DEF = {'plus': {'TP', 'FN'}, 'minus': {'TN', 'FP'},
             'pred': {'TP', 'FP'}, 'nopred': {'FN', 'TN'},   # decisione del modello (include FP)
             'TP': {'TP'}, 'FN': {'FN'}, 'TN': {'TN'}, 'FP': {'FP'}}
CONTRASTS = [('TP', 'TN'), ('pred', 'TN')]   # TP vs TN (modello corretto) + (TP+FP) vs TN (incl. falsi positivi)
COMBINED_ABS = True   # True: combined = attn*|grad| ; False: attn*ReLU(grad)
EXPLAIN_FILE = os.path.join(SPLIT_DIR, 'explain_top3.parquet')   # stato TP/FN/TN/FP; toggle: explain_top5.parquet
import re as _re
_m = _re.search(r'top\d+', os.path.basename(EXPLAIN_FILE))
TAG = ('_' + _m.group(0)) if _m else ''   # es '_top3' -> appeso a tutti gli output
TAG = TAG + ('_absC' if COMBINED_ABS else '')   # output separati
print('TAG output =', TAG or '(nessuno)')
EXPLAIN = pd.read_parquet(EXPLAIN_FILE)
EXPLAINS = {v: pd.read_parquet(os.path.join(SPLIT_DIR, f'explain_{v}.parquet')) for v in ['top3', 'top5']}
VERSIONS = list(EXPLAINS)   # ogni test viene fatto per entrambe le versioni
# ALL tumors and signatures present in the file (dynamic, no longer hardcoded)
TISSUES = {tt.lower(): {'targets': sorted(EXPLAIN.loc[EXPLAIN['tumor_type'] == tt, 'signature'].unique())}
           for tt in sorted(EXPLAIN['tumor_type'].unique())}
print('explain:', os.path.basename(EXPLAIN_FILE), EXPLAIN.shape, '| tumori:', len(TISSUES), '| contrasti:', CONTRASTS)
print({t: c['targets'] for t, c in TISSUES.items()})

## 1. Helpers: subset cache, plus/minus labels, per-patient metrics
`spearman_to_target` -> per case, mean correlation (over slides) between T's ranking and every other
signature. `topk_jaccard_uniqueness` -> mean Jaccard and uniqueness of T's top-K, per case.


In [ ]:
def build_cache(tissue):
    '''Costruisce la cache pooled del tessuto dai parquet grad-score dei 4 fold (una slide alla volta).'''
    import glob
    cs = set(EXPLAIN.loc[EXPLAIN['tumor_type'].str.upper() == tissue.upper(), 'Case ID'])
    parqs = sorted(glob.glob(os.path.join(FOLD_DIR, 'val_gradcam_fold[1-4]_*.parquet')))
    out = os.path.join(OUT_DIR, f'attr_{tissue}_subset_scores.parquet'); writer = None; nsl = 0
    for pp in parqs:
        pf = pq.ParquetFile(pp)
        for rg in range(pf.metadata.num_row_groups):
            cid = pf.read_row_group(rg, columns=['case_id']).column('case_id')[0].as_py()
            if cid in cs:
                t = pf.read_row_group(rg)
                if writer is None:
                    writer = pq.ParquetWriter(out, t.schema, compression='snappy')
                writer.write_table(t); nsl += 1
    if writer is not None:
        writer.close()
    print(f'[{tissue}] cache costruita: {nsl} slide da {len(parqs)} fold parquet')

def split_path(tissue, target):
    return os.path.join(SPLIT_DIR, f'{tissue}_{target}{TAG}.csv')

def load_subset(tissue):
    '''Read the tissue parquet cache: USECOLS only, strings as category (lower memory).'''
    pth = os.path.join(OUT_DIR, f'attr_{tissue}_subset_scores.parquet')
    if not os.path.exists(pth):
        build_cache(tissue)
    names = pq.ParquetFile(pth).schema_arrow.names
    cols = [c for c in USECOLS if c in names]
    tbl = pq.read_table(pth, columns=cols)
    for c in CAT_COLS:
        if c in tbl.column_names:
            i = tbl.schema.get_field_index(c)
            tbl = tbl.set_column(i, c, tbl.column(c).dictionary_encode())
    a = tbl.to_pandas()
    g = a['grad_score'].astype('float32')
    if 'attention_weight' in a.columns:
        a['combined_score'] = (a['attention_weight'].astype('float32') *
                               (np.abs(g) if COMBINED_ABS else np.clip(g, 0, None))).astype('float32')
        a = a.drop(columns=['attention_weight'])
    a['grad_score'] = np.abs(g)
    print(f'[{tissue}] {os.path.basename(pth)}: {a.shape}  mem={a.memory_usage(deep=True).sum()/1e9:.2f} GB')
    return a

def build_all_missing_caches():
    '''Build the missing caches for ALL tumors in ONE pass (routing each slide to its tumor).'''
    import glob
    tumor_of = dict(zip(EXPLAIN['Case ID'], EXPLAIN['tumor_type'].str.lower()))
    missing = [t for t in TISSUES if not os.path.exists(os.path.join(OUT_DIR, f'attr_{t}_subset_scores.parquet'))]
    if not missing:
        print('cache: tutte gia presenti'); return
    print('costruisco cache per', len(missing), 'tumori:', missing, flush=True)
    writers = {}
    for pp in sorted(glob.glob(os.path.join(FOLD_DIR, 'val_gradcam_fold[1-4]_*.parquet'))):
        pf = pq.ParquetFile(pp)
        for rg in range(pf.metadata.num_row_groups):
            cid = pf.read_row_group(rg, columns=['case_id']).column('case_id')[0].as_py()
            t = tumor_of.get(cid)
            if t in missing:
                tbl = pf.read_row_group(rg)
                if t not in writers:
                    writers[t] = pq.ParquetWriter(os.path.join(OUT_DIR, f'attr_{t}_subset_scores.parquet'),
                                                  tbl.schema, compression='snappy')
                writers[t].write_table(tbl)
    for w in writers.values():
        w.close()
    print('cache costruite:', sorted(writers.keys()), flush=True)

def labels_for(tissue, target, ver=None):
    '''From explain_topN.parquet (one row per case,signature): group=plus if the signature is in true_topN,
    eligible=yes se in pred_topN. Stato: plus&yes=TP, plus&no=FN, minus&yes=FP (falsi positivi), minus&no=TN.'''
    ver = ver if ver is not None else VERSIONS[-1]
    ex = EXPLAINS[ver]
    sub = ex[(ex['tumor_type'].str.upper() == tissue.upper()) & (ex['signature'] == target)]
    present = dict(zip(sub['Case ID'], np.where(sub['group'] == 'plus', 'present', 'absent')))
    status = {}
    for cid, g_, el in zip(sub['Case ID'], sub['group'], sub['eligible']):
        if g_ == 'plus':
            status[cid] = 'TP' if el == 'yes' else 'FN'
        else:
            status[cid] = 'FP' if el == 'yes' else 'TN'
    return present, status

def status_of(tissue, target, index, ver=None):
    _, status = labels_for(tissue, target, ver)
    return pd.Series({c: status.get(c) for c in index})

def grp(s, st, label):
    return s[st.isin(GROUP_DEF[label])].dropna()   # scarta NaN (gruppi vuoti / Spearman indefinito su combined)

def spearman_to_target(attr, target, score=SCORE_COL):
    '''Per case: Spearman(rank score T, rank score S) for every S!=T, averaged over slides.'''
    case_of = attr.groupby('sample_id', observed=True)['case_id'].first().astype(str)
    rows = {}
    for sid, g in attr.groupby('sample_id', observed=True):
        piv = g.pivot(index='patch_index', columns='signature_name', values=score)
        if target not in piv.columns:
            continue
        r = piv.rank()
        rows[str(sid)] = r.corrwith(r[target]).drop(labels=[target])
    R = pd.DataFrame(rows).T
    R.columns = [str(c) for c in R.columns]
    R['case_id'] = [case_of.get(s) for s in R.index]
    return R.groupby('case_id').mean()


def spearman_all_targets(attr, targets, score):
    """UNA passata sulle slide: matrice 30x30 di Spearman (corrcoef sui ranghi), media per caso."""
    all_sigs = sorted(attr['signature_name'].astype(str).unique())
    case_of  = attr.groupby('sample_id', observed=True)['case_id'].first().astype(str)
    per_case = {}
    for sid, g in attr.groupby('sample_id', observed=True):
        piv = g.pivot(index='patch_index', columns='signature_name', values=score)
        piv.columns = [str(x) for x in piv.columns]
        piv = piv.reindex(columns=all_sigs)
        per_case.setdefault(case_of[sid], []).append(np.corrcoef(piv.rank().values, rowvar=False))
    cases = sorted(per_case)
    M = np.stack([np.nanmean(np.stack(per_case[c]), axis=0) for c in cases])
    idx = {x: i for i, x in enumerate(all_sigs)}
    out = {}
    for t in targets:
        if t not in idx: continue
        others = [x for x in all_sigs if x != t]
        out[t] = pd.DataFrame(M[:, idx[t], [idx[x] for x in others]], index=cases, columns=others)
    return out

def topk_all_targets(attr, targets, K, score):
    """ONE pass over the slides: top-K of every signature computed once and reused for each target."""
    case_of = attr.groupby('sample_id', observed=True)['case_id'].first().astype(str)
    acc = {t: ({}, {}) for t in targets}
    for sid, g in attr.groupby('sample_id', observed=True):
        tops = {str(x): set(gs.nlargest(K, score)['patch_index'])
                for x, gs in g.groupby('signature_name', observed=True)}
        for t in targets:
            T = tops.get(t)
            if not T: continue
            others = [v for x, v in tops.items() if x != t and v]
            js = [len(T & o) / len(T | o) for o in others]
            union_o = set().union(*others) if others else set()
            acc[t][0][str(sid)] = float(np.mean(js)) if js else np.nan
            acc[t][1][str(sid)] = len(T - union_o) / len(T)
    out = {}
    for t in targets:
        df = pd.DataFrame({'jaccard': pd.Series(acc[t][0]), 'uniqueness': pd.Series(acc[t][1])})
        if df.empty: continue
        df['case_id'] = [case_of.get(x) for x in df.index]
        out[t] = df.groupby('case_id').mean()
    return out

def topk_jaccard_uniqueness(attr, target, K, score=SCORE_COL):
    '''Per case: mean Jaccard of T's top-K against every other signature, plus T's uniqueness.'''
    case_of = attr.groupby('sample_id', observed=True)['case_id'].first().astype(str)
    jac, uniq = {}, {}
    for sid, g in attr.groupby('sample_id', observed=True):
        tops = {str(s): set(gs.nlargest(K, score)['patch_index'])
                for s, gs in g.groupby('signature_name', observed=True)}
        if target not in tops or not tops[target]:
            continue
        T = tops[target]
        others = [v for s, v in tops.items() if s != target and v]
        js = [len(T & o) / len(T | o) for o in others]
        jac[str(sid)] = float(np.mean(js)) if js else np.nan
        union_o = set().union(*others) if others else set()
        uniq[str(sid)] = len(T - union_o) / len(T)
    df = pd.DataFrame({'jaccard': pd.Series(jac), 'uniqueness': pd.Series(uniq)})
    df['case_id'] = [case_of.get(s) for s in df.index]
    return df.groupby('case_id').mean()

def box_contrast(ax, s, st, gA, gB, ylabel, alt):
    '''Boxplot gruppo A vs B (definiti su stato), punti colorati per stato; ritorna (a, b, p).'''
    a = grp(s, st, gA); b = grp(s, st, gB)
    p = mannwhitneyu(a, b, alternative=alt).pvalue if len(a) and len(b) else np.nan
    try:
        bp = ax.boxplot([a.values, b.values], tick_labels=[gA, gB], widths=0.5,
                        patch_artist=True, showfliers=False)
    except TypeError:
        bp = ax.boxplot([a.values, b.values], labels=[gA, gB], widths=0.5,
                        patch_artist=True, showfliers=False)
    for patch, col in zip(bp['boxes'], ['#8fd19e', '#f1a6a6']):
        patch.set_facecolor(col); patch.set_alpha(0.6)
    for i, lab in enumerate([gA, gB], 1):
        sub = grp(s, st, lab)
        ax.scatter(rng.normal(i, 0.06, len(sub)), sub.values,
                   c=[colors[st[c]] for c in sub.index], s=28, zorder=3, edgecolor='k', linewidth=0.3)
    ax.set_ylabel(ylabel)
    return a, b, p

## 2. Carico/costruisco i subset per COAD e STAD

In [ ]:
build_all_missing_caches()
DATA = {}
for tissue in list(TISSUES):
    try:
        attr = load_subset(tissue)
    except Exception as e:
        print(f'[{tissue}] SKIP (nessuna attribuzione): {type(e).__name__}'); TISSUES.pop(tissue, None); continue
    DATA[tissue] = attr
    print(f'[{tissue}] casi={attr["case_id"].nunique()} slide={attr["sample_id"].nunique()} '
          f'targets={len(TISSUES[tissue]["targets"])}', flush=True)

## 3. Spearman: target ranking vs other signatures - **TP vs TN**, grad vs combined
Primary contrast TP vs TN; every metric computed with both `grad_score` and `combined_score`
(= attn * ReLU(grad)) - see the `score` column in the table. The plots show the first score
(grad_score). `spearman_specificity_ALL.csv` reports both scores and contrasts.


In [ ]:
gA, gB = CONTRASTS[0]   # TP vs TN (primario)
all_rows = []
_SPEAR = {}
for tissue in TISSUES:
    for _sc in SCORES:
        _SPEAR[(tissue, _sc)] = spearman_all_targets(DATA[tissue], TISSUES[tissue]['targets'], _sc)
    print(f'  spearman calcolato: {tissue}', flush=True)

for tissue in TISSUES:
    attr = DATA[tissue]
    for target in TISSUES[tissue]['targets']:
        for score in SCORES:
            Rc = _SPEAR[(tissue, score)].get(target)
            if Rc is None: continue
            mean_corr = Rc.mean(axis=1)
            st = status_of(tissue, target, mean_corr.index)
            if PLOT_PER_TARGET and score == SCORES[0]:
                fig, axes = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={'width_ratios': [1, 1.6]})
                a, b, p = box_contrast(axes[0], mean_corr, st, gA, gB,
                                       f'Spearman media (T={target} vs altre)', alt='less')
                axes[0].set_title(f'{tissue.upper()} {target} [{score}] - corr media\nMWU p({gA}<{gB})={p:.3g}  n={len(a)}/{len(b)}')
                axes[0].legend(handles=[Line2D([0],[0], marker='o', ls='', mec='k', mfc=colors[s2], label=s2)
                                        for s2 in ['TP','FN','TN','FP']], title='stato', fontsize=7)
                maskA = st.isin(GROUP_DEF[gA]); maskB = st.isin(GROUP_DEF[gB])
                per_sig = pd.DataFrame({gA: Rc[maskA].mean(), gB: Rc[maskB].mean()}).sort_values(gB)
                y = np.arange(len(per_sig)); h = 0.4
                axes[1].barh(y + h/2, per_sig[gA], height=h, color='#2ca02c', alpha=0.8, label=gA)
                axes[1].barh(y - h/2, per_sig[gB], height=h, color='#d62728', alpha=0.8, label=gB)
                axes[1].set_yticks(y); axes[1].set_yticklabels(per_sig.index, fontsize=7)
                axes[1].set_xlabel('Spearman media a T'); axes[1].legend()
                axes[1].set_title(f'corr verso ciascuna signature ({gA} vs {gB})')
                plt.tight_layout()
                plt.savefig(os.path.join(OUT_DIR, f'{tissue}_{target}_spearman_TPvsTN{TAG}.png'), dpi=150, bbox_inches='tight')
                plt.show()
            for ver in VERSIONS:
                stv = status_of(tissue, target, mean_corr.index, ver)
                for (cA, cB) in CONTRASTS:
                    x = grp(mean_corr, stv, cA); z = grp(mean_corr, stv, cB)
                    pv = mannwhitneyu(x, z, alternative='less').pvalue if len(x) and len(z) else np.nan
                    all_rows.append({'version': ver, 'tissue': tissue, 'target': target, 'score': score,
                                     'contrast': f'{cA}<{cB}', 'nA': len(x), 'nB': len(z),
                                     'meanA': round(x.mean(), 4), 'meanB': round(z.mean(), 4),
                                     'delta': round(x.mean() - z.mean(), 4), 'MWU_p': round(pv, 4)})
spearman_summary = pd.DataFrame(all_rows)
spearman_summary.to_csv(os.path.join(OUT_DIR, f'spearman_specificity_ALL{TAG}.csv'), index=False)
spearman_summary

## 3b. **Per-pair** Spearman: target vs EACH signature (TP vs TN)
Averaging over the 29 signatures can hide a decorrelation towards specific ones. Here the
correlations are kept separate: for each (target, other signature) we compare TP vs TN (one-sided
MWU, TP<TN). Saves `spearman_perpair_ALL.csv` and shows the pairs with p < 0.1.


In [ ]:
def per_case_corr_matrix(attr):
    """Per case: 30x30 Spearman matrix between signatures (corrcoef on ranks), averaged over slides."""
    case_of = attr.groupby('sample_id', observed=True)['case_id'].first().astype(str)
    sigs = None; per = {}
    for sid, g in attr.groupby('sample_id', observed=True):
        piv = g.pivot(index='patch_index', columns='signature_name', values=SCORE_COL)
        sigs = list(piv.columns)
        R = np.corrcoef(piv.rank().values, rowvar=False)
        per.setdefault(case_of[sid], []).append(R)
    cases = sorted(per)
    M = np.stack([np.mean(per[c], axis=0) for c in cases])
    return cases, sigs, M

pp_rows = []
for tissue in TISSUES:
    cases, sigs, M = per_case_corr_matrix(DATA[tissue]); sidx = {s: i for i, s in enumerate(sigs)}
    for target in TISSUES[tissue]['targets']:
        if target not in sidx:
            continue
        ti = sidx[target]; st = status_of(tissue, target, cases).values
        for s in sigs:
            if s == target:
                continue
            v = M[:, ti, sidx[s]]; a = v[st == 'TP']; b = v[st == 'TN']
            pv = mannwhitneyu(a, b, alternative='less').pvalue if len(a) and len(b) else np.nan
            pp_rows.append({'tissue': tissue, 'target': target, 'other': s,
                            'corr_TP': round(float(a.mean()), 3) if len(a) else np.nan,
                            'corr_TN': round(float(b.mean()), 3) if len(b) else np.nan,
                            'delta': round(float(a.mean() - b.mean()), 3) if len(a) and len(b) else np.nan,
                            'p_TP<TN': round(float(pv), 4)})
perpair = pd.DataFrame(pp_rows)
perpair.to_csv(os.path.join(OUT_DIR, f'spearman_perpair_ALL{TAG}.csv'), index=False)
perpair[perpair['p_TP<TN'] < 0.1].sort_values('p_TP<TN')

## 4. Jaccard e uniqueness dei top-K - **TP vs TN**, grad vs combined
Jaccard (atteso TP<TN) e uniqueness (atteso TP>TN), per `grad_score` e `combined_score`.
`jaccard_uniqueness_ALL.csv` copre score x K x contrasti.

In [ ]:
gA, gB = CONTRASTS[0]
ju_rows = []
_TOPK = {}
for tissue in TISSUES:
    for _sc in SCORES:
        for _K in K_LIST:
            _TOPK[(tissue, _sc, _K)] = topk_all_targets(DATA[tissue], TISSUES[tissue]['targets'], _K, _sc)
    print(f'  topK calcolato: {tissue}', flush=True)

for tissue in TISSUES:
    attr = DATA[tissue]
    for target in TISSUES[tissue]['targets']:
        for score in SCORES:
            dfK = {K: _TOPK[(tissue, score, K)].get(target) for K in K_LIST}
            if any(v is None for v in dfK.values()): continue
            for K in K_LIST:
                d = dfK[K]; st = status_of(tissue, target, d.index)
                for metric, alt in [('jaccard', 'less'), ('uniqueness', 'greater')]:
                    s = d[metric]
                    for ver in VERSIONS:
                        stv = status_of(tissue, target, d.index, ver)
                        for (cA, cB) in CONTRASTS:
                            x = grp(s, stv, cA); z = grp(s, stv, cB)
                            pv = mannwhitneyu(x, z, alternative=alt).pvalue if len(x) and len(z) else np.nan
                            ju_rows.append({'version': ver, 'tissue': tissue, 'target': target, 'score': score,
                                            'metric': metric, 'K': K, 'contrast': f'{cA}vs{cB}', 'alt': alt,
                                            'nA': len(x), 'nB': len(z), 'meanA': round(x.mean(), 4),
                                            'meanB': round(z.mean(), 4), 'delta': round(x.mean() - z.mean(), 4),
                                            'MWU_p': round(pv, 4)})
            if PLOT_PER_TARGET and score == SCORES[0]:
                fig, axes = plt.subplots(2, len(K_LIST), figsize=(4 * len(K_LIST), 8), squeeze=False)
                for j, K in enumerate(K_LIST):
                    d = dfK[K]; st = status_of(tissue, target, d.index)
                    _, _, pj = box_contrast(axes[0][j], d['jaccard'], st, gA, gB, 'Jaccard', alt='less')
                    axes[0][j].set_title(f'Jaccard K={K}\np({gA}<{gB})={pj:.3g}', fontsize=9)
                    _, _, pu = box_contrast(axes[1][j], d['uniqueness'], st, gA, gB, 'uniqueness', alt='greater')
                    axes[1][j].set_title(f'uniq K={K}\np({gA}>{gB})={pu:.3g}', fontsize=9)
                fig.suptitle(f'{tissue.upper()} {target} - Jaccard (riga 1) / uniqueness (riga 2), {gA} vs {gB} per K', y=1.01)
                plt.tight_layout()
                plt.savefig(os.path.join(OUT_DIR, f'{tissue}_{target}_jaccard_uniqueness_TPvsTN{TAG}.png'), dpi=150, bbox_inches='tight')
                plt.show()
ju_summary = pd.DataFrame(ju_rows)
ju_summary.to_csv(os.path.join(OUT_DIR, f'jaccard_uniqueness_ALL{TAG}.csv'), index=False)
ju_summary[(ju_summary['contrast'] == 'TPvsTN') & (ju_summary['score'] == SCORES[0])]

## 5. Reading the result
- **Primary contrast TP vs TN.** Compares the cases where the model **detects the signature
  correctly** (TP) with those where it **correctly excludes it** (TN): this isolates the model's
  behaviour when it is consistent, removing the noise of its errors (FN/FP). If T is specific, in
  the TP group Spearman and Jaccard are lower and uniqueness is higher.
- **plus vs minus** (in the table) is the biological contrast, more conservative since it mixes
  TP + FN.
- **Status (TP/FN/TN/FP)** comes from group (signature in the true top-3) x eligible (signature in
  the predicted top-3).
- **Caveats.** Some TP groups are small (SBS93, SBS8), so the p-values are unstable and a
  multiple-comparison correction is needed.


## 6. Significant combinations: **p < 0.054** (~0.05) — STANDALONE cell
Collects every combination (tumor, signature, score, K, contrast, metric) with `MWU_p < 0.054` from
**Spearman** and **Jaccard/uniqueness**, sorted by increasing p. Re-reads the CSVs from disk, so it
works on a fresh kernel. Rows where one group has fewer than `NMIN` cases are flagged as unreliable.
Saves `notebook1_significativi_p05.csv`.


In [ ]:
# ---- STANDALONE: ALL combinations with p ~<= 0.05 (notebook 1) ----
import os
import numpy as np, pandas as pd
from IPython.display import display

OUT_DIR_ = globals().get('OUT_DIR', '../grad/xai_outputs')
import re as _re
TAG = globals().get('TAG')
if TAG is None:
    _ef = globals().get('EXPLAIN_FILE', 'explain_top3.parquet')
    _m = _re.search(r'top\d+', os.path.basename(str(_ef)))
    TAG = ('_' + _m.group(0)) if _m else ''
print('TAG =', TAG or '(nessuno)')
NMIN     = 10   # 'affidabile' = ENTRAMBI i gruppi >= NMIN pazienti (flag, non filtra)

frames = []
fsp = os.path.join(OUT_DIR_, f'spearman_specificity_ALL{TAG}.csv')
if os.path.exists(fsp):
    sp = pd.read_csv(fsp); sp['metrica'] = 'spearman'; sp['K'] = np.nan
    frames.append(sp[['tissue','target','metrica','score','K','contrast','nA','nB','delta','MWU_p']])
fju = os.path.join(OUT_DIR_, f'jaccard_uniqueness_ALL{TAG}.csv')
if os.path.exists(fju):
    ju = pd.read_csv(fju).rename(columns={'metric': 'metrica'})
    frames.append(ju[['tissue','target','metrica','score','K','contrast','nA','nB','delta','MWU_p']])

allr = pd.concat(frames, ignore_index=True)
allr['MWU_p'] = pd.to_numeric(allr['MWU_p'], errors='coerce')

hits = allr[allr['MWU_p'].round(2) <= 0.05].copy()    # 0.054 arrotonda a 0.05 -> incluso
hits['n_min'] = hits[['nA', 'nB']].min(axis=1)
hits['affidabile'] = np.where(hits['n_min'] >= NMIN, 'si', 'NO')
hits = hits.sort_values('MWU_p').reset_index(drop=True)

print(f'combinazioni con p arrotondato <= 0.05: {len(hits)} su {len(allr)}'
      f'   | affidabili: {(hits["affidabile"]=="si").sum()}   | gruppo piccolo: {(hits["affidabile"]=="NO").sum()}')
pd.set_option('display.max_rows', 500)
print('\n================= TUTTE le combinazioni (ordinate per p) =================')
display(hits)

out = os.path.join(OUT_DIR_, f'notebook1_significativi_p05{TAG}.csv')
hits.to_csv(out, index=False)
print('\nsalvato:', out)

## 7. Summary figure — **pred vs TN** contrast (publication style)
Dot matrix: rows = significant (tumor, signature) pairs, columns = **K**, one panel per **metric**.
Dot **size** = significance (-log10 p); **colour** = effect size, already oriented in the expected
direction (lower Jaccard/Spearman, higher uniqueness = more specific).
STANDALONE cell: it re-reads the CSVs, the notebook does not need to be re-run.


In [ ]:
# ---- STANDALONE: figura riassuntiva pred vs TN ----
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

OUT_DIR_ = globals().get('OUT_DIR', '../grad/xai_outputs')
import re as _re
TAG = globals().get('TAG')
if TAG is None:
    _ef = globals().get('EXPLAIN_FILE', 'explain_top3.parquet')
    _m = _re.search(r'top\d+', os.path.basename(str(_ef)))
    TAG = ('_' + _m.group(0)) if _m else ''
print('TAG =', TAG or '(nessuno)')

# ---------- dati ----------
frames = []
fsp = os.path.join(OUT_DIR_, f'spearman_specificity_ALL{TAG}.csv')
if os.path.exists(fsp):
    sp = pd.read_csv(fsp); sp['metrica'] = 'Spearman'; sp['K'] = np.nan
    frames.append(sp[['tissue','target','metrica','score','K','contrast','nA','nB','delta','MWU_p']])
fju = os.path.join(OUT_DIR_, f'jaccard_uniqueness_ALL{TAG}.csv')
if os.path.exists(fju):
    ju = pd.read_csv(fju).rename(columns={'metric': 'metrica'})
    ju['metrica'] = ju['metrica'].str.capitalize()
    frames.append(ju[['tissue','target','metrica','score','K','contrast','nA','nB','delta','MWU_p']])
d = pd.concat(frames, ignore_index=True)
d['MWU_p'] = pd.to_numeric(d['MWU_p'], errors='coerce')

d = d[d['contrast'].isin(['pred<TN', 'predvsTN'])]          # contrasto pred vs TN
d = d[d['MWU_p'].round(2) <= 0.05].copy()                   # p arrotondato <= 0.05
if d.empty:
    raise SystemExit('nessuna combinazione significativa per pred vs TN')

# effetto orientato: positivo = piu' specifico
d['effect'] = np.where(d['metrica'].eq('Uniqueness'), d['delta'], -d['delta'])
d['negp']   = -np.log10(d['MWU_p'].clip(lower=1e-6))
d['lab']    = d['tissue'].str.upper() + '  ' + d['target']

labs  = d.groupby('lab')['MWU_p'].min().sort_values(ascending=False).index.tolist()
ypos  = {l: i for i, l in enumerate(labs)}
mets  = [m for m in ['Jaccard', 'Uniqueness', 'Spearman'] if m in set(d['metrica'])]
Ks    = sorted(d.loc[d['metrica'] != 'Spearman', 'K'].dropna().unique())

# ---------- stile ----------
plt.rcParams.update({'font.size': 9, 'axes.linewidth': 0.6,
                     'xtick.major.width': 0.6, 'ytick.major.width': 0.6})
widths = [len(Ks) if m != 'Spearman' else 1.15 for m in mets]
fig, axes = plt.subplots(1, len(mets), sharey=True,
                         figsize=(1.05 * sum(widths) + 5.5, 0.30 * len(labs) + 2.2),
                         gridspec_kw={'width_ratios': widths})
axes = np.atleast_1d(axes)
vmax = float(np.nanmax(d['effect'])); vmin = 0.0
cmap = plt.cm.YlGnBu

for ax, met in zip(axes, mets):
    sub = d[d['metrica'] == met]
    if met == 'Spearman':
        xs, xt, xl = [0] * len(sub), [0], ['tutte le\npatch']
        x = np.zeros(len(sub))
    else:
        xmap = {k: i for i, k in enumerate(Ks)}
        x = sub['K'].map(xmap).values
        xt, xl = range(len(Ks)), [f'{int(k)}' for k in Ks]
    y = sub['lab'].map(ypos).values
    s = 25 + 165 * (sub['negp'] / max(d['negp'].max(), 1e-9))
    ax.scatter(x, y, s=s, c=sub['effect'], cmap=cmap, vmin=vmin, vmax=vmax,
               edgecolor='0.25', linewidth=0.45, zorder=3)
    ax.set_xticks(list(xt)); ax.set_xticklabels(xl)
    ax.set_title(met, fontsize=11, pad=8)
    ax.set_xlim(-0.6, (len(Ks) - 0.4) if met != 'Spearman' else 0.6)
    ax.grid(axis='y', color='0.9', lw=0.6, zorder=0)
    for sp_ in ('top', 'right', 'left'): ax.spines[sp_].set_visible(False)
    ax.tick_params(axis='y', length=0)
    if met != 'Spearman': ax.set_xlabel('top-K patch', labelpad=6)

axes[0].set_yticks(range(len(labs))); axes[0].set_yticklabels(labs, fontsize=8)
axes[0].set_ylim(-0.8, len(labs) - 0.2)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin, vmax))
cb = fig.colorbar(sm, ax=axes.tolist(), fraction=0.02, pad=0.02)
cb.set_label('ampiezza effetto (direzione: piu\' specifico $\\rightarrow$)', fontsize=9)
cb.outline.set_linewidth(0.5)

handles = [Line2D([], [], marker='o', ls='', markerfacecolor='0.75', markeredgecolor='0.25',
                  markersize=np.sqrt(25 + 165 * f), label=lab)
           for f, lab in [(0.15, 'p ~ 0.05'), (0.55, 'p ~ 0.01'), (1.0, 'p < 0.001')]]
axes[-1].legend(handles=handles, title='significativita\'', loc='center left',
                bbox_to_anchor=(1.28, 0.5), frameon=False, fontsize=8, title_fontsize=9,
                labelspacing=1.1, borderpad=0.8)

fig.suptitle('Specificita\' spaziale delle firme — contrasto  pred (TP+FP)  vs  TN',
             fontsize=13, y=0.995)
out = os.path.join(OUT_DIR_, f'figura_pred_vs_TN{TAG}.png')
plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(out.replace(f'{TAG}.png', f'{TAG}.pdf'), bbox_inches='tight', facecolor='white')
plt.show()
print('salvata:', out, '(+ .pdf)')
print(f'combinazioni: {d["lab"].nunique()} | metriche: {mets} | punti: {len(d)}')

## 8. Tumor x signature grid — **shape = K**, score = `combined_score`
**(B)** single panel: four sub-columns per tumor, one per K, each with a different shape.
**(F)** faceted: one panel per K. Colour = -log10(p).
`METRICA` is `jaccard` (default) or `uniqueness`; Spearman uses all patches and therefore has no K.


In [ ]:
# ---- STANDALONE: griglia tumore x firma — shape = K, score = combined ----
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

OUT_DIR_  = globals().get('OUT_DIR', '../grad/xai_outputs')
import re as _re
TAG = globals().get('TAG')
if TAG is None:
    _ef = globals().get('EXPLAIN_FILE', 'explain_top3.parquet')
    _m = _re.search(r'top\d+', os.path.basename(str(_ef)))
    TAG = ('_' + _m.group(0)) if _m else ''
print('TAG =', TAG or '(nessuno)')
SCORE_SEL = 'combined_score'   # score fissato
METRICA   = 'uniqueness'          # 'jaccard' | 'uniqueness'  (hanno K)  | 'spearman' (nessun K)
CONTRAST  = 'TP'             # 'pred' (TP+FP vs TN)  oppure  'TP'

# ---------- dati ----------
frames = []
fsp = os.path.join(OUT_DIR_, f'spearman_specificity_ALL{TAG}.csv')
if os.path.exists(fsp):
    sp = pd.read_csv(fsp); sp['metrica'] = 'spearman'; sp['K'] = np.nan
    frames.append(sp[['tissue','target','metrica','score','K','contrast','nA','nB','delta','MWU_p']])
fju = os.path.join(OUT_DIR_, f'jaccard_uniqueness_ALL{TAG}.csv')
if os.path.exists(fju):
    ju = pd.read_csv(fju).rename(columns={'metric': 'metrica'})
    frames.append(ju[['tissue','target','metrica','score','K','contrast','nA','nB','delta','MWU_p']])
d = pd.concat(frames, ignore_index=True)
d['MWU_p'] = pd.to_numeric(d['MWU_p'], errors='coerce')

d = d[(d['metrica'].str.lower() == METRICA) & (d['score'] == SCORE_SEL)]
d = d[d['contrast'].isin([f'{CONTRAST}<TN' if METRICA == 'spearman' else f'{CONTRAST}vsTN'])]
d = d[d['MWU_p'].round(2) <= 0.05].copy()
if d.empty:
    raise SystemExit(f'nessun risultato per {METRICA} / {SCORE_SEL} / {CONTRAST}')

# ---- level = K (shape). Spearman has no K -> a single 'all patches' level ----
if METRICA == 'spearman':
    d['lvl'] = 'tutte'; lvls = ['tutte']
else:
    d['lvl'] = d['K'].astype(int).astype(str)
    lvls = [str(int(k)) for k in sorted(d['K'].dropna().unique())]
lvl_name = 'n. patch (K)'

d['negp'] = -np.log10(d['MWU_p'].clip(lower=1e-6))
tums = sorted(d['tissue'].str.upper().unique())
sigs = d.groupby('target').size().sort_values().index.tolist()
xi = {t: i for i, t in enumerate(tums)}; yi = {s: i for i, s in enumerate(sigs)}
mk = {l: m for l, m in zip(lvls, ['o', 's', '^', 'D', 'v'])}
vmin, vmax = 0.0, float(d['negp'].max())
cmap = plt.cm.viridis
plt.rcParams.update({'font.size': 9})

def _scatter(ax, sub, spread=True):
    for l in lvls:
        s2 = sub[sub['lvl'] == l]
        if s2.empty: continue
        off = (lvls.index(l) - (len(lvls) - 1) / 2) * (0.66 / max(len(lvls), 1)) if spread else 0
        ax.scatter(s2['tissue'].str.upper().map(xi) + off, s2['target'].map(yi),
                   c=s2['negp'], cmap=cmap, vmin=vmin, vmax=vmax, marker=mk[l],
                   s=55, edgecolor='0.25', linewidth=0.5, zorder=3)
    ax.set_xticks(range(len(tums))); ax.set_xticklabels(tums, rotation=90, fontsize=7)
    ax.set_yticks(range(len(sigs))); ax.set_yticklabels(sigs, fontsize=7)
    ax.set_xlim(-0.7, len(tums) - 0.3); ax.set_ylim(-0.8, len(sigs) - 0.2)
    ax.grid(color='0.92', lw=0.5, zorder=0)
    for s_ in ('top', 'right'): ax.spines[s_].set_visible(False)

# ---------- Layout B ----------
fig, ax = plt.subplots(figsize=(0.55 * len(tums) + 5, 0.30 * len(sigs) + 2.5))
_scatter(ax, d)
ax.set_title(f'{METRICA.capitalize()} ({SCORE_SEL}) — firme significative per tumore, '
             f'contrasto {CONTRAST} vs TN\nshape = {lvl_name}', fontsize=11)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin, vmax))
plt.colorbar(sm, ax=ax, fraction=0.02, pad=0.01).set_label('-log10(p-value)')
ax.legend(handles=[Line2D([], [], marker=mk[l], ls='', color='0.35', markersize=7, label=f'K = {l}')
                   for l in lvls], title=lvl_name, loc='upper left',
          bbox_to_anchor=(1.06, 1), frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR_, f'griglia_B_{METRICA}_{SCORE_SEL}_{CONTRAST}{TAG}.png'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# ---------- Layout sfaccettato per K ----------
ncol = 2 if len(lvls) > 1 else 1
nrow = int(np.ceil(len(lvls) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * (0.45 * len(tums) + 3.2),
                                              nrow * (0.26 * len(sigs) + 2.2)), squeeze=False)
for a, l in zip(axes.flat, lvls):
    _scatter(a, d[d['lvl'] == l], spread=False)
    a.set_title(f'K = {l}', fontsize=11)
for a in axes.flat[len(lvls):]: a.axis('of')
fig.suptitle(f'{METRICA.capitalize()} ({SCORE_SEL}) — firme significative per tumore, '
             f'sfaccettato per numero di patch', fontsize=13, y=1.0)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin, vmax))
fig.colorbar(sm, ax=axes.ravel().tolist(), fraction=0.015, pad=0.015).set_label('-log10(p-value)')
plt.savefig(os.path.join(OUT_DIR_, f'griglia_facet_{METRICA}_{SCORE_SEL}_{CONTRAST}{TAG}.png'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'{METRICA} / {SCORE_SEL} / {CONTRAST}: {len(d)} punti | '
      f'{d["tissue"].nunique()} tumori | {d["target"].nunique()} firme | K: {lvls}')

## 9. List of ~20 significant images (different tumors) and their patches — to download
**STANDALONE** cell. Among the reliable significant results it picks **one combination per tumor**
(different tumors), takes a **TP case** and its **top-N patches** (coordinates) from the attribution
cache. It also attaches the **GDC file_id** (from the metadata) for direct download.
Saves `immagini_da_scaricare.csv` (the slides) and `patches_da_guardare.csv` (the coordinates).


In [ ]:
# ---- STANDALONE: 20 slide significative (tumori diversi) + top-patch, pronte da scaricare ----
import os, json, glob
import numpy as np, pandas as pd, pyarrow.parquet as pq
from IPython.display import display

OUT_DIR  = '../grad/xai_outputs'
SPLIT_DIR = '../coad_stad_explainability'
META_JSON = glob.glob('../metadata.repository*.json')   # per il file_id GDC (facoltativo)

VER      = 'top5'            # versione etichette (top3/top5)
SCORE    = 'combined_score'  # score usato per rankare le patch
METRICA  = 'jaccard'         # metrica per selezionare i significativi
CONTR    = 'predvsTN'        # 'predvsTN' oppure 'TPvsTN'
N_IMG    = 20                # quante slide (tumori diversi)
N_PATCH  = 12                # patch per slide
NMIN     = 10                # affidabilita' minima dei gruppi

# 1) risultati significativi (prova i nomi file in ordine)
cands = [f'jaccard_uniqueness_ALL{s}.csv' for s in ['_top5_absC', f'_{VER}_absC', f'_{VER}', '_absC', '']]
RES = next((os.path.join(OUT_DIR, c) for c in cands if os.path.exists(os.path.join(OUT_DIR, c))), None)
assert RES, 'nessun jaccard_uniqueness_ALL*.csv trovato'
d = pd.read_csv(RES)
if 'version' in d.columns: d = d[d['version'] == VER]
d = d[(d['score'] == SCORE) & (d['metric'] == METRICA) & (d['contrast'] == CONTR)]
d = d[d['MWU_p'].round(2) <= 0.05]
d = d[d[['nA', 'nB']].min(axis=1) >= NMIN]
best = d.sort_values('MWU_p').drop_duplicates('tissue').head(N_IMG)   # una per tumore
print(f'file risultati: {os.path.basename(RES)} | combinazioni scelte: {len(best)} tumori')

# 2) slide -> GDC file_id map (optional)
fid = {}
if META_JSON:
    for rec in json.load(open(META_JSON[0])):
        fn = rec.get('file_name', '')
        if fn.endswith('.svs'): fid[fn[:-4]] = rec.get('file_id', '')

# 3) TP dal file explain
ex = pd.read_parquet(os.path.join(SPLIT_DIR, f'explain_{VER}.parquet'))
def tp_cases(tissue, target):
    sub = ex[(ex['tumor_type'].str.upper() == tissue.upper()) & (ex['signature'] == target)]
    return set(sub.loc[(sub['group'] == 'plus') & (sub['eligible'] == 'yes'), 'Case ID'])

# 4) for each combination: the TP case with the strongest signal + its top-N patches
COLS = ['case_id', 'feature_file', 'signature_name', 'coord_x', 'coord_y',
        'original_index', 'grad_score', 'attention_weight']
img_rows, patch_rows = [], []
for _, r in best.iterrows():
    tissue, target = r['tissue'], r['target']
    tps = tp_cases(tissue, target)
    cache = os.path.join(OUT_DIR, f'attr_{tissue}_subset_scores.parquet')
    if not tps or not os.path.exists(cache):
        print(f'  skip {tissue} {target} (TP={len(tps)}, cache={os.path.exists(cache)})'); continue
    pf = pq.ParquetFile(cache); parts = []
    for rg in range(pf.metadata.num_row_groups):        # legge solo le slide dei casi TP
        cid = pf.read_row_group(rg, columns=['case_id']).column('case_id')[0].as_py()
        if cid in tps:
            t = pf.read_row_group(rg, columns=COLS).to_pandas()
            parts.append(t[t['signature_name'].astype(str) == target])
    if not parts: continue
    t = pd.concat(parts, ignore_index=True)
    g = t['grad_score'].astype('float32')
    t['combined_score'] = (t['attention_weight'].astype('float32') * np.abs(g)).astype('float32')
    case = t.groupby('case_id')['combined_score'].max().idxmax()   # caso TP col picco piu' alto
    tc = t[t['case_id'] == case]
    slide = str(tc['feature_file'].iloc[0])[:-3] if str(tc['feature_file'].iloc[0]).endswith('.pt') else str(tc['feature_file'].iloc[0])
    top = tc.nlargest(N_PATCH, 'combined_score')
    img_rows.append({'tumor': tissue.upper(), 'signature': target, 'case_id': str(case),
                     'slide': slide, 'gdc_file_id': fid.get(slide, ''),
                     'metrica': METRICA, 'contrast': CONTR, 'p_value': r['MWU_p'],
                     'nA': int(r['nA']), 'nB': int(r['nB']), 'n_patch': len(top)})
    for rk, (_, pr) in enumerate(top.iterrows(), 1):
        patch_rows.append({'tumor': tissue.upper(), 'signature': target, 'case_id': str(case), 'slide': slide,
                           'rank': rk, 'coord_x': int(pr['coord_x']), 'coord_y': int(pr['coord_y']),
                           'original_index': int(pr['original_index']),
                           'combined_score': round(float(pr['combined_score']), 6)})

IMG   = pd.DataFrame(img_rows)
PATCH = pd.DataFrame(patch_rows)
IMG.to_csv(os.path.join(OUT_DIR, 'immagini_da_scaricare.csv'), index=False)
PATCH.to_csv(os.path.join(OUT_DIR, 'patches_da_guardare.csv'), index=False)
print(f'\n{len(IMG)} slide da {IMG["tumor"].nunique()} tumori | {len(PATCH)} patch totali')
print('salvati: immagini_da_scaricare.csv  e  patches_da_guardare.csv\n')
display(IMG)
print('\n--- example patches from the first slide ---')
display(PATCH[PATCH['slide'] == IMG['slide'].iloc[0]] if len(IMG) else PATCH)